# 💳 Enterprise Fraud Detection — Cost-Sensitive LightGBM Pipeline
### Table 1: Machine Learning & Explainable AI Engine (Hari's Pipeline)
---
This self-contained Google Colab notebook trains a **Cost-Sensitive LightGBM model** with **asymmetric financial penalties ($10 \times \text{FN} + 1 \times \text{FP}$)** and generates **TreeSHAP local explanations** for the Samsung Capstone Human-in-the-Loop (HITL) platform.

## 1. Setup & Dependencies
Mount Google Drive (optional, if your datasets are stored in Drive) and install required dependencies.

In [ ]:
# Optional: Mount Google Drive if your data is stored in Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully!')
except Exception:
    print('Running locally or Drive already mounted.')

# Install packages
!pip install -q lightgbm shap scikit-learn matplotlib

## 2. Dataset Discovery & Ingestion
Locate `train_transaction.csv` and `train_identity.csv` across common Colab and Drive directories.

In [ ]:
import os, sys, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix, roc_curve

warnings.filterwarnings('ignore')

# Search common locations
def find_csv(filename):
    candidates = [
        Path('/content') / filename,
        Path('/content/drive/MyDrive') / filename,
        Path('/content/drive/MyDrive/ieee-fraud-detection') / filename,
        Path('/content/drive/MyDrive/IEEE-CIS') / filename,
        Path.cwd() / filename,
    ]
    for c in candidates:
        if c.is_file(): return c
    for p in Path('/content').rglob(filename):
        if p.is_file(): return p
    return None

txn_path = find_csv('train_transaction.csv')
id_path = find_csv('train_identity.csv')
print('Transaction file:', txn_path)
print('Identity file:   ', id_path)

assert txn_path and id_path, 'Please upload or specify train_transaction.csv and train_identity.csv'

## 3. Data Preprocessing & Memory Optimization
Merge datasets, prune ultra-sparse V-columns to save RAM, encode categoricals, and sort chronologically.

In [ ]:
print('Loading and merging datasets...')
txn_df = pd.read_csv(txn_path)
id_df = pd.read_csv(id_path)
df = pd.merge(txn_df, id_df, on='TransactionID', how='left')
del txn_df, id_df
gc.collect()

# Prune ultra-sparse V-columns
v_drop = [f'V{i}' for i in range(35, 279) if f'V{i}' in df.columns]
df.drop(columns=v_drop, inplace=True)

# Chronological sort
df.sort_values('TransactionDT', inplace=True)
df.reset_index(drop=True, inplace=True)

# Feature Engineering
df['TransactionDay'] = (df['TransactionDT'] // (24 * 3600)).astype(np.int16)
df['TransactionHour'] = ((df['TransactionDT'] // 3600) % 24).astype(np.int8)
if 'card1' in df.columns:
    c_mean = df.groupby('card1')['TransactionAmt'].transform('mean')
    df['TransactionAmt_to_mean_card1'] = (df['TransactionAmt'] / (c_mean + 1.0)).astype(np.float32)

# Categorical encoding
cat_cols = df.select_dtypes(include=['object']).columns
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes.astype(np.int32)

# Downcast floats
for c in df.select_dtypes(include=['float64']).columns:
    df[c] = df[c].astype(np.float32)

print(f'Final preprocessed shape: {df.shape} | Fraud rate: {df["isFraud"].mean()*100:.2f}%')

## 4. Custom Cost-Sensitive Objective Function
Implement the asymmetric financial penalty matrix ($C_{FN}=10.0, C_{FP}=1.0$) directly into LightGBM's gradient and hessian calculations.

In [ ]:
C_FP = 1.0
C_FN = 10.0

def cost_sensitive_objective(y_pred, dtrain):
    y_true = dtrain.get_label().astype(np.float64)
    p = 1.0 / (1.0 + np.exp(-np.clip(y_pred, -15.0, 15.0)))
    weights = np.where(y_true == 1, C_FN, C_FP)
    grad = weights * (p - y_true)
    hess = np.maximum(weights * p * (1.0 - p), 1e-8)
    return grad, hess

def cost_sensitive_eval(y_pred, dtrain):
    y_true = dtrain.get_label().astype(int)
    p = 1.0 / (1.0 + np.exp(-np.clip(y_pred, -15.0, 15.0)))
    y_hat = (p >= 0.5).astype(int)
    fp = int(np.sum((y_hat == 1) & (y_true == 0)))
    fn = int(np.sum((y_hat == 0) & (y_true == 1)))
    total_cost = C_FP * fp + C_FN * fn
    return 'total_cost', total_cost, False

## 5. Chronological Split & Booster Training
Train on the earliest 80% and evaluate on the untouched latest 20%.

In [ ]:
split = int(len(df) * 0.80)
train_df = df.iloc[:split]
test_df = df.iloc[split:]

features = [c for c in df.columns if c not in ['TransactionID', 'isFraud', 'TransactionDT']]
X_train, y_train = train_df[features], train_df['isFraud'].values
X_test, y_test = test_df[features], test_df['isFraud'].values
test_ids = test_df['TransactionID'].values

dtrain = lgb.Dataset(X_train, label=y_train)
dtest = lgb.Dataset(X_test, label=y_test, reference=dtrain)

params = {
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.80,
    'bagging_fraction': 0.80,
    'bagging_freq': 1,
    'verbosity': -1,
    'seed': 42
}

print('Training Cost-Sensitive Booster...')
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1500,
    valid_sets=[dtrain, dtest],
    valid_names=['train', 'valid'],
    fobj=cost_sensitive_objective,
    feval=cost_sensitive_eval,
    callbacks=[lgb.early_stopping(stopping_rounds=80, verbose=True)]
)

## 6. Financial Evaluation & Benchmark Curves

In [ ]:
raw_preds = model.predict(X_test, num_iteration=model.best_iteration)
y_prob = 1.0 / (1.0 + np.exp(-np.clip(raw_preds, -15.0, 15.0)))

auc = roc_auc_score(y_test, y_prob)
print(f'Test ROC-AUC: {auc:.4f}')

# Evaluate at 0.5 vs Optimal Threshold
y_hat_05 = (y_prob >= 0.5).astype(int)
cm_05 = confusion_matrix(y_test, y_hat_05)
tn, fp, fn, tp = cm_05.ravel()
cost = C_FP * fp + C_FN * fn
print(f'Total Financial Cost at 0.50 threshold: {cost:,} (FP={fp:,}, FN={fn:,})')

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {auc:.4f}', color='#2563eb', lw=2)
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Cost-Sensitive LightGBM — Final Test ROC')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. TreeSHAP Explainability & Handoff Export
Compute exact local TreeSHAP attributions and export `lightgbm_shap_output.json` for ingestion into the FastAPI & Streamlit HITL console.

In [ ]:
os.makedirs('outputs', exist_ok=True)
n_explain = min(10000, len(X_test))
X_exp = X_test.iloc[:n_explain]
ids_exp = test_ids[:n_explain]
probs_exp = y_prob[:n_explain]

print(f'Computing TreeSHAP contributions for {n_explain:,} transactions...')
contribs = model.predict(X_exp, num_iteration=model.best_iteration, pred_contrib=True)
shap_mat = contribs[:, :-1]

records = []
for i in range(n_explain):
    top_idx = np.argsort(np.abs(shap_mat[i]))[-3:][::-1]
    records.append({
        'transaction_id': str(ids_exp[i]),
        'dataset_name': 'ieee_cis',
        'prediction_probability': round(float(probs_exp[i]), 6),
        'top3_shap_features': {features[j]: round(float(shap_mat[i, j]), 6) for j in top_idx}
    })

with open('outputs/lightgbm_shap_output.json', 'w') as f:
    json.dump(records, f, indent=2)
print('[SUCCESS] Exported outputs/lightgbm_shap_output.json')

# Summary plot
mean_shap = np.mean(np.abs(shap_mat), axis=0)
top20 = np.argsort(mean_shap)[-20:]
plt.figure(figsize=(10, 8))
plt.barh([features[j] for j in top20], mean_shap[top20], color='#2563eb')
plt.xlabel('Mean |TreeSHAP| Contribution')
plt.title('Global Feature Importance')
plt.tight_layout()
plt.savefig('outputs/shap_summary.png', dpi=160)
plt.show()
print('[SUCCESS] Saved outputs/shap_summary.png')

# Download files if in Colab
try:
    from google.colab import files
    files.download('outputs/lightgbm_shap_output.json')
    files.download('outputs/shap_summary.png')
except Exception:
    print('Saved to outputs/ directory.')